# TAREA 6
María Fernanda Gómez Flores  
Carlos Hassan Hernández Rohan  
Portafolios de Inversión

## PUNTO A

In [65]:
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt
from scipy.optimize import minimize

### 1. MÍNIMA SEMIVARIANZA

In [66]:
# Descarga de Datos
prices = yf.download(['PG', 'COST', 'KO', 'WMT', 'CLX', 'KHC'],
                     start='2016-01-01', end='2026-10-15')['Close']

prices.isna().sum()

/var/folders/x9/60hcc2ln62d3r3xyv90d3wl40000gn/T/ipykernel_16539/1304681109.py:2: FutureWarning: YF.download() has changed argument auto_adjust default to True
  prices = yf.download(['PG', 'COST', 'KO', 'WMT', 'CLX', 'KHC'],
[*********************100%***********************]  6 of 6 completed


Ticker
CLX     0
COST    0
KHC     0
KO      0
PG      0
WMT     0
dtype: int64

In [67]:
# Calculo de rendimientos
rets=prices.pct_change().dropna()

In [68]:
# Comprobar datos
print(prices.index.min(),prices.index.max())
print(prices.isna().sum())

2016-01-04 00:00:00 2026-10-14 00:00:00
Ticker
CLX     0
COST    0
KHC     0
KO      0
PG      0
WMT     0
dtype: int64


In [69]:
def opt_min_semivar(prices: pd.DataFrame):
    #Calcular los rendimientos
    rets = prices.pct_change().dropna()

    #Calcular matriz de correlación
    corr = rets.corr()

    #Calcular downside risk
    below_zero = (rets[rets<0]).fillna(0)
    downside = np.array(below_zero.std())

    #Calcular matriz de semivarianza
    n_assets = len(prices.columns)
    semivar_matrix = (downside.reshape(n_assets,1) @ downside.reshape(1,n_assets))*corr

    # Funcion.
    semivar = lambda w: w.T @ semivar_matrix @ w

    #Variables iniciales (pesos)
    w_0 = np.ones(n_assets) / n_assets

    # Restricciones.
    restr = lambda w: sum(w) - 1

    #rangos de las variables de decisión
    rangos = [(0,1)] * n_assets

    # tolerancia
    tol = 1e-12

    #Optimizar
    results = minimize(fun=semivar, x0=w_0, bounds=rangos,
                       constraints={'fun': restr, 'type': 'eq'}, tol=tol)

    if not results.success:
        print(results.message)

    return results.x

In [70]:
# Verificar resultados de la función
w_min_semivar=opt_min_semivar(prices)
dict(zip(prices.columns,w_min_semivar))

{'CLX': np.float64(0.08781196206031486),
 'COST': np.float64(0.15076650660278745),
 'KHC': np.float64(0.01768757190803307),
 'KO': np.float64(0.32799181106453945),
 'PG': np.float64(0.22265933057144988),
 'WMT': np.float64(0.19308281779287542)}

#### Comprobación Montecarlo

In [71]:
below_zero=(rets[rets<0]).fillna(0)
below_zero

Ticker,CLX,COST,KHC,KO,PG,WMT
Date,,,,,,
2016-01-05,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2016-01-06,-0.000473,-0.009254,0.000000,-0.005405,-0.009667,0.000000
2016-01-07,-0.010942,-0.022910,-0.022168,-0.016541,-0.008734,0.000000
2016-01-08,0.000000,-0.017504,-0.010014,-0.002643,-0.015678,-0.022912
2016-01-11,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...
2026-10-08,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2026-10-09,0.000000,-0.001055,-0.009342,0.000000,0.000000,0.000000
2026-10-12,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [72]:
downside=np.array(below_zero.std())
downside

array([0.00998865, 0.0084987 , 0.01128803, 0.00725476, 0.0073185 ,
       0.00839042])

In [73]:
n_assets=len(prices.columns)
#downside @ downside.T
semivar_matrix=(downside.reshape(n_assets,1) @ downside.reshape(1,n_assets))*rets.corr()
semivar_matrix

Ticker,CLX,COST,KHC,KO,PG,WMT
Ticker,,,,,,
CLX,0.000100,0.000027,0.000038,0.000027,0.000038,0.000025
COST,0.000027,0.000072,0.000026,0.000024,0.000026,0.000041
KHC,0.000038,0.000026,0.000127,0.000038,0.000037,0.000026
KO,0.000027,0.000024,0.000038,0.000053,0.000033,0.000021
PG,0.000038,0.000026,0.000037,0.000033,0.000054,0.000026
WMT,0.000025,0.000041,0.000026,0.000021,0.000026,0.000070


In [74]:
semivar_port=lambda w: w.T @ semivar_matrix @ w

In [75]:
# Obtener semivarianza mínima
semivar_port(w_min_semivar)

np.float64(3.541707959742086e-05)

In [76]:
# Simular portafolios con montecarlo
np.random.seed(42)
n_sim=10000
w_random=np.random.dirichlet(np.ones(n_assets), size=n_sim)

In [77]:
# Obtener la mínima semivarianza
semivars_port=[semivar_port(w_i) for w_i in w_random]
np.min(semivars_port)

np.float64(3.563832002011221e-05)

In [78]:
# Obtener los pesos del portafolio con mínima semivarianza
w_semivar=w_random[np.argmin(semivars_port)]
dict(zip(prices.columns, w_semivar))

{'CLX': np.float64(0.04523198269375449),
 'COST': np.float64(0.18032970063786),
 'KHC': np.float64(0.02431282278193316),
 'KO': np.float64(0.34349231541965103),
 'PG': np.float64(0.19048463084264106),
 'WMT': np.float64(0.2161485476241604)}

In [79]:
# Comparar resultados
comparacion=pd.DataFrame({
    'SciPy':w_min_semivar*100,
    'Montecarlo':w_semivar*100
},index=prices.columns)

comparacion

,SciPy,Montecarlo
Ticker,,
CLX,8.781196,4.523198
COST,15.076651,18.032970
KHC,1.768757,2.431282
KO,32.799181,34.349232
PG,22.265933,19.048463
WMT,19.308282,21.614855


In [80]:
# Comparar semivarianza mínima
resultados_semivar=pd.DataFrame({
    'Semivarianza':[semivar_port(w_min_semivar),np.min(semivars_port)]
},index=['SciPy','Montecarlo'])
resultados_semivar

,Semivarianza
SciPy,0.000035
Montecarlo,0.000036


**Conclusión**  
En mínima semivarianza obtuvimos 0.0000354171 con SciPy y 0.0000356383 con Montecarlo, entonces los resultados quedaron bastante cercanos. En SciPy el mayor peso fue KO con 32.80%, seguido de PG con 22.27% y WMT con 19.31%. En Montecarlo también KO tuvo el mayor peso con 34.35%. SciPy encontró una semivarianza un poco menor porque Montecarlo solo compara los portafolios que simulamos.

### 2. MÁXIMO OMEGA

In [81]:
def opt_max_omega(prices: pd.DataFrame):
    #Calcular los rendimientos
    rets = prices.pct_change().dropna()

    #Calcular downside y upside risk
    below_zero=rets[rets<0].fillna(0)
    above_zero=rets[rets>0].fillna(0)
    upside_risk=np.array(above_zero.std())
    downside_risk=np.array(below_zero.std())

    # Calcular Omega individual
    omega=upside_risk/downside_risk

    # Funcion
    func_omega=lambda w: -sum(w*omega)

    # Restricciones
    restr=lambda w: sum(w)-1

    # Valor Inicial
    n_assets=len(prices.columns)
    w_0=np.ones(n_assets)/n_assets

    # tol
    tol=1e-12

    # Rangos
    rangos=[(0,1)]*n_assets

    # optimización
    results=minimize(fun=func_omega,x0=w_0,bounds=rangos,
                     constraints={'fun':restr,'type':'eq'},tol=tol)

    if not results.success:
        print(results.message)

    return results.x

In [82]:
# Obtener los pesos
w_max_omega=opt_max_omega(prices)
dict(zip(prices.columns,w_max_omega))

{'CLX': np.float64(0.0),
 'COST': np.float64(0.0),
 'KHC': np.float64(0.0),
 'KO': np.float64(0.0),
 'PG': np.float64(3.677613769070831e-16),
 'WMT': np.float64(1.0)}

#### Comprobación Montecarlo

In [83]:
below_zero=rets[rets<0].fillna(0)
above_zero=rets[rets>0].fillna(0)

In [84]:
upside_risk=np.array(above_zero.std())
upside_risk

array([0.00907326, 0.00843026, 0.01005745, 0.0069553 , 0.0074014 ,
       0.00898112])

In [85]:
downside_risk=np.array(below_zero.std())
downside_risk

array([0.00998865, 0.0084987 , 0.01128803, 0.00725476, 0.0073185 ,
       0.00839042])

In [86]:
omega=upside_risk/downside_risk
omega

array([0.90835702, 0.99194668, 0.89098311, 0.95872162, 1.0113267 ,
       1.07040147])

In [87]:
# Obtener el maximo omega
func_omega=lambda w: sum(w*omega)

In [88]:
# Simular portafolios con montecarlo
w_random=np.random.dirichlet(np.ones(n_assets), size=n_sim)

In [89]:
# Obtener el maximo omega
omegas_port=[func_omega(w_i) for w_i in w_random]
np.max(omegas_port)

np.float64(1.0574780436342488)

In [90]:
# Obtener los pesos del portafolio con maximo omega
w_omega_mc=w_random[np.argmax(omegas_port)]
dict(zip(rets.columns,w_omega_mc))

{'CLX': np.float64(0.03399832366440288),
 'COST': np.float64(0.012029943661879684),
 'KHC': np.float64(0.01443209502964946),
 'KO': np.float64(0.033455007736670934),
 'PG': np.float64(0.002450246946761601),
 'WMT': np.float64(0.9036343829606355)}

In [91]:
# Comparar resultados
comparacion_omega=pd.DataFrame({
    'SciPy':w_max_omega*100,
    'Montecarlo':w_omega_mc*100
},index=prices.columns)
comparacion_omega

,SciPy,Montecarlo
Ticker,,
CLX,0.000000e+00,3.399832
COST,0.000000e+00,1.202994
KHC,0.000000e+00,1.443210
KO,0.000000e+00,3.345501
PG,3.677614e-14,0.245025
WMT,1.000000e+02,90.363438


In [92]:
# Comparar Omega máximo
resultados_omega=pd.DataFrame({
    'Omega':[func_omega(w_max_omega),np.max(omegas_port)]
},index=['SciPy','Montecarlo'])
resultados_omega

,Omega
SciPy,1.070401
Montecarlo,1.057478


**Conclusión**  
En máximo Omega cambió mucho la distribución, porque con SciPy WMT quedó con prácticamente 100% del portafolio, mientras que en Montecarlo obtuvo 90.36%. El Omega máximo fue 1.070401 con SciPy y 1.057478 con Montecarlo. Esto tiene sentido porque WMT fue el activo con el Omega individual más alto, pero el portafolio quedó muy concentrado en una sola empresa. Aquí estamos usando la fórmula de Omega que vimos en clase.

### 3. SEMIVARIANZA OBJETIVO

In [93]:
# Descarga de Precios del Benchmark
benchmark = yf.download('KXI', start='2016-01-01', end='2026-10-15')['Close']
benchmark

/var/folders/x9/60hcc2ln62d3r3xyv90d3wl40000gn/T/ipykernel_16539/4160543330.py:2: FutureWarning: YF.download() has changed argument auto_adjust default to True
  benchmark = yf.download('KXI', start='2016-01-01', end='2026-10-15')['Close']
[*********************100%***********************]  1 of 1 completed


Ticker,KXI
Date,
2016-01-04,35.621929
2016-01-05,35.711323
2016-01-06,35.396488
2016-01-07,34.933975
2016-01-08,34.654140
...,...
2026-10-08,67.309998
2026-10-09,67.400002
2026-10-12,67.889999


In [94]:
# Obtener rendimientos del benchmark
rets_benchmark=benchmark.pct_change().dropna()
rets_benchmark

Ticker,KXI
Date,
2016-01-05,0.002510
2016-01-06,-0.008816
2016-01-07,-0.013067
2016-01-08,-0.008010
2016-01-11,0.008636
...,...
2026-10-08,0.021086
2026-10-09,0.001337
2026-10-12,0.007270


In [95]:
# Comprobar fechas en común
fechas=rets.index.intersection(rets_benchmark.index)
print(len(rets),len(rets_benchmark),len(fechas))
print(fechas.min(),fechas.max())
print(rets.loc[fechas].isna().sum())
print(rets_benchmark.loc[fechas].isna().sum())

2710 2710 2710
2016-01-05 00:00:00 2026-10-14 00:00:00
Ticker
CLX     0
COST    0
KHC     0
KO      0
PG      0
WMT     0
dtype: int64
Ticker
KXI    0
dtype: int64


In [96]:
def opt_target_semivar(prices: pd.DataFrame, benchmark: pd.Series):
    #Calcular los rendimientos
    rets=prices.pct_change().dropna()
    rets_benchmark=benchmark.pct_change().dropna()

    # Sustraer rendimiento del benchmark de rendimientos individuales
    alpha=rets.sub(rets_benchmark.squeeze(),axis=0).dropna()

    # Separar los rendimientos por debajo de cero
    below_zero_t=(alpha[alpha<0]).fillna(0)

    # Calcular target downside risk
    downside_risk_t=np.array(below_zero_t.std())

    # Calcular la matriz de target semivariance
    corr=alpha.corr()
    n_assets=len(prices.columns)
    semivar_matrix=(downside_risk_t.reshape(n_assets,1) @ downside_risk_t.reshape(1,n_assets))*corr

    # Funcion
    func_target=lambda w: w.T @ semivar_matrix @ w

    # Restricciones
    restr=lambda w: sum(w)-1

    # Valor Inicial
    w_0=np.ones(n_assets)/n_assets

    # tol
    tol=1e-12

    # Rangos
    rangos=[(0,1)]*n_assets

    # optimización
    results=minimize(fun=func_target,x0=w_0,bounds=rangos,
                     constraints={'fun':restr,'type':'eq'},tol=tol)

    if not results.success:
        print(results.message)

    return results.x

In [97]:
# Obtener los pesos del portafolio con mínima semivarianza target
w_target_semivar=opt_target_semivar(prices,benchmark)
dict(zip(prices.columns,w_target_semivar))

{'CLX': np.float64(0.041431713501744245),
 'COST': np.float64(0.15737476382556778),
 'KHC': np.float64(0.07722149635710172),
 'KO': np.float64(0.3359963210493366),
 'PG': np.float64(0.27466408548670607),
 'WMT': np.float64(0.11331161977954358)}

#### Comprobación Montecarlo

In [98]:
# Sustraer rendimiento del benchmark de rendimientos individuales
alpha=rets.sub(rets_benchmark.squeeze(),axis=0).dropna()
alpha

Ticker,CLX,COST,KHC,KO,PG,WMT
Date,,,,,,
2016-01-05,0.011938,-0.000065,0.006981,0.001028,0.000681,0.021246
2016-01-06,0.008344,-0.000438,0.010724,0.003411,-0.000851,0.018828
2016-01-07,0.002124,-0.009843,-0.009101,-0.003474,0.004333,0.036355
2016-01-08,0.012309,-0.009493,-0.002003,0.005367,-0.007667,-0.014902
2016-01-11,0.003251,0.008523,0.006677,-0.006949,0.000579,0.002066
...,...,...,...,...,...,...
2026-10-08,-0.006124,-0.015069,0.001662,0.001636,-0.002347,0.001103
2026-10-09,0.002977,-0.002392,-0.010679,0.001853,0.002913,0.005808
2026-10-12,-0.000707,0.005952,0.000813,0.012491,0.005757,0.008267


In [99]:
# Separar los rendimientos por debajo de cero
below_zero_t=(alpha[alpha<0]).fillna(0)
below_zero_t

Ticker,CLX,COST,KHC,KO,PG,WMT
Date,,,,,,
2016-01-05,0.000000,-0.000065,0.000000,0.000000,0.000000,0.000000
2016-01-06,0.000000,-0.000438,0.000000,0.000000,-0.000851,0.000000
2016-01-07,0.000000,-0.009843,-0.009101,-0.003474,0.000000,0.000000
2016-01-08,0.000000,-0.009493,-0.002003,0.000000,-0.007667,-0.014902
2016-01-11,0.000000,0.000000,0.000000,-0.006949,0.000000,0.000000
...,...,...,...,...,...,...
2026-10-08,-0.006124,-0.015069,0.000000,0.000000,-0.002347,0.000000
2026-10-09,0.000000,-0.002392,-0.010679,0.000000,0.000000,0.000000
2026-10-12,-0.000707,0.000000,0.000000,0.000000,0.000000,0.000000


In [100]:
# Calcular target downside risk
downside_risk_t=np.array(below_zero_t.std())
downside_risk_t

array([0.00865549, 0.00684588, 0.00956584, 0.00473383, 0.00478275,
       0.00712828])

In [101]:
# Calcular la matriz de target semivariance
n_assets=len(prices.columns)
#downside @ downside.T
semivar_matrix_t=(downside_risk_t.reshape(n_assets,1) @ downside_risk_t.reshape(1,n_assets))*alpha.corr()
semivar_matrix_t

Ticker,CLX,COST,KHC,KO,PG,WMT
Ticker,,,,,,
CLX,7.491748e-05,0.000003,0.000007,7.313672e-07,0.000011,0.000004
COST,2.607500e-06,0.000047,-0.000006,-2.740260e-06,-0.000001,0.000018
KHC,7.057973e-06,-0.000006,0.000092,3.781574e-06,0.000002,-0.000003
KO,7.313672e-07,-0.000003,0.000004,2.240916e-05,0.000003,-0.000003
PG,1.118005e-05,-0.000001,0.000002,2.808965e-06,0.000023,0.000001
WMT,4.002851e-06,0.000018,-0.000003,-3.128373e-06,0.000001,0.000051


In [102]:
# Simular portafolios con montecarlo
w_random=np.random.dirichlet(np.ones(n_assets), size=n_sim)

In [103]:
# Obtener la mínima target semivariance
func_target = lambda w: w.T @ semivar_matrix_t @ w
mts=[func_target(w_i)for w_i in w_random]
np.min(mts)

np.float64(7.876657086226663e-06)

In [104]:
# Obtener los pesos del portafolio con mínima semivarianza target
w_tarjet_semivar = w_random[np.argmin(mts)]
dict(zip(rets.columns, w_tarjet_semivar))

{'CLX': np.float64(0.05018925389310219),
 'COST': np.float64(0.13631386488141561),
 'KHC': np.float64(0.07003453981745411),
 'KO': np.float64(0.3520023944212645),
 'PG': np.float64(0.25385946443983465),
 'WMT': np.float64(0.13760048254692883)}

In [105]:
# Comparar resultados
comparacion_target=pd.DataFrame({
    'SciPy':w_target_semivar*100,
    'Montecarlo':w_tarjet_semivar*100
},index=prices.columns)

comparacion_target

,SciPy,Montecarlo
Ticker,,
CLX,4.143171,5.018925
COST,15.737476,13.631386
KHC,7.722150,7.003454
KO,33.599632,35.200239
PG,27.466409,25.385946
WMT,11.331162,13.760048


In [106]:
# Comparar semivarianza objetivo
resultados_target=pd.DataFrame({
    'Semivarianza objetivo':[func_target(w_target_semivar),np.min(mts)]
},index=['SciPy','Montecarlo'])
resultados_target.apply(lambda x: x.map(lambda v: f'{v:.10f}'))

,Semivarianza objetivo
SciPy,0.0000078267
Montecarlo,0.0000078767


**Conclusión**  
En semivarianza objetivo utilizamos KXI como benchmark. Con SciPy obtuvimos 0.0000078267 y con Montecarlo 0.0000078767, por lo que SciPy encontró una semivarianza objetivo ligeramente menor. En SciPy los activos con mayor peso fueron KO con 33.60% y PG con 27.47%; en Montecarlo fueron KO con 35.20% y PG con 25.39%. Los pesos cambiaron respecto a mínima semivarianza porque ahora tomamos como referencia el rendimiento de KXI.

## PUNTO B

Seleccionamos cinco activos de consumo básico: PG, COST, KO, WMT y CLX. Utilizamos KXI como benchmark porque corresponde al sector de consumo básico. Vamos a aplicar las tres funciones del Punto A.

In [107]:
# Descarga de Datos
prices_b=yf.download(['PG','COST','KO','WMT','CLX'],
                     start='2016-01-01',end='2026-10-15')['Close']
prices_b.isna().sum()

/var/folders/x9/60hcc2ln62d3r3xyv90d3wl40000gn/T/ipykernel_16539/273862728.py:2: FutureWarning: YF.download() has changed argument auto_adjust default to True
  prices_b=yf.download(['PG','COST','KO','WMT','CLX'],
[*********************100%***********************]  5 of 5 completed


Ticker
CLX     0
COST    0
KO      0
PG      0
WMT     0
dtype: int64

In [108]:
# Calculo de rendimientos
rets_b=prices_b.pct_change().dropna()
rets_b

Ticker,CLX,COST,KO,PG,WMT
Date,,,,,
2016-01-05,0.014448,0.002445,0.003537,0.003190,0.023755
2016-01-06,-0.000473,-0.009254,-0.005405,-0.009667,0.010012
2016-01-07,-0.010942,-0.022910,-0.016541,-0.008734,0.023289
2016-01-08,0.004298,-0.017504,-0.002643,-0.015678,-0.022912
2016-01-11,0.011887,0.017159,0.001687,0.009215,0.010702
...,...,...,...,...,...
2026-10-08,0.014962,0.006017,0.022722,0.018739,0.022189
2026-10-09,0.004314,-0.001055,0.003190,0.004250,0.007145
2026-10-12,0.006563,0.013222,0.019761,0.013027,0.015537


### 1. MÍNIMA SEMIVARIANZA

In [109]:
# Obtener los pesos
w_min_semivar_b=opt_min_semivar(prices_b)
dict(zip(prices_b.columns,w_min_semivar_b))

{'CLX': np.float64(0.0865988047263096),
 'COST': np.float64(0.15434667701099575),
 'KO': np.float64(0.3330872367015388),
 'PG': np.float64(0.2347644041269832),
 'WMT': np.float64(0.1912028774341727)}

In [110]:
# Calcular matriz de correlación
corr_b=rets_b.corr()
corr_b

Ticker,CLX,COST,KO,PG,WMT
Ticker,,,,,
CLX,1.000000,0.312793,0.369140,0.520421,0.302428
COST,0.312793,1.000000,0.394182,0.424521,0.568327
KO,0.369140,0.394182,1.000000,0.613858,0.350033
PG,0.520421,0.424521,0.613858,1.000000,0.429701
WMT,0.302428,0.568327,0.350033,0.429701,1.000000


In [111]:
below_zero_b=(rets_b[rets_b<0]).fillna(0)
below_zero_b

Ticker,CLX,COST,KO,PG,WMT
Date,,,,,
2016-01-05,0.000000,0.000000,0.000000,0.000000,0.000000
2016-01-06,-0.000473,-0.009254,-0.005405,-0.009667,0.000000
2016-01-07,-0.010942,-0.022910,-0.016541,-0.008734,0.000000
2016-01-08,0.000000,-0.017504,-0.002643,-0.015678,-0.022912
2016-01-11,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...
2026-10-08,0.000000,0.000000,0.000000,0.000000,0.000000
2026-10-09,0.000000,-0.001055,0.000000,0.000000,0.000000
2026-10-12,0.000000,0.000000,0.000000,0.000000,0.000000


In [112]:
downside_b=np.array(below_zero_b.std())
downside_b

array([0.00998865, 0.0084987 , 0.00725476, 0.0073185 , 0.00839042])

In [113]:
# Calcular matriz de semivarianza
n_assets_b=len(prices_b.columns)
semivar_matrix_b=(downside_b.reshape(n_assets_b,1) @ downside_b.reshape(1,n_assets_b))*corr_b
semivar_matrix_b

Ticker,CLX,COST,KO,PG,WMT
Ticker,,,,,
CLX,0.000100,0.000027,0.000027,0.000038,0.000025
COST,0.000027,0.000072,0.000024,0.000026,0.000041
KO,0.000027,0.000024,0.000053,0.000033,0.000021
PG,0.000038,0.000026,0.000033,0.000054,0.000026
WMT,0.000025,0.000041,0.000021,0.000026,0.000070


In [114]:
# Obtener semivarianza mínima
semivar_port_b=lambda w: w.T @ semivar_matrix_b @ w
semivar_port_b(w_min_semivar_b)

np.float64(3.545754537972392e-05)

### 2. MÁXIMO OMEGA

In [115]:
# Obtener los pesos
w_max_omega_b=opt_max_omega(prices_b)
dict(zip(prices_b.columns,w_max_omega_b))

{'CLX': np.float64(3.3514211068838184e-16),
 'COST': np.float64(5.290270139111073e-16),
 'KO': np.float64(1.7902713190223769e-16),
 'PG': np.float64(8.881784197001252e-16),
 'WMT': np.float64(0.9999999999999981)}

In [116]:
# Calcular upside risk
above_zero_b=rets_b[rets_b>0].fillna(0)
upside_risk_b=np.array(above_zero_b.std())
upside_risk_b

array([0.00907326, 0.00843026, 0.0069553 , 0.0074014 , 0.00898112])

In [117]:
# Calcular Omega individual
omega_b=upside_risk_b/downside_b
omega_b

array([0.90835702, 0.99194668, 0.95872162, 1.0113267 , 1.07040147])

In [118]:
# Obtener máximo Omega
func_omega_b=lambda w: sum(w*omega_b)
func_omega_b(w_max_omega_b)

np.float64(1.0704014710362366)

### 3. SEMIVARIANZA OBJETIVO

In [119]:
# Usar el mismo benchmark del Punto A
benchmark_b=benchmark

In [120]:
# Obtener rendimientos del benchmark
rets_benchmark_b=rets_benchmark

In [121]:
# Verificar fechas en común
fechas_b=rets_b.index.intersection(rets_benchmark_b.index)
print(len(rets_b),len(rets_benchmark_b),len(fechas_b))
print(prices_b.isna().sum())

2710 2710 2710
Ticker
CLX     0
COST    0
KO      0
PG      0
WMT     0
dtype: int64


In [122]:
# Obtener los pesos
w_target_semivar_b=opt_target_semivar(prices_b,benchmark_b)
dict(zip(prices_b.columns,w_target_semivar_b))

{'CLX': np.float64(0.052305201455112554),
 'COST': np.float64(0.15801241334763902),
 'KO': np.float64(0.3725141199885424),
 'PG': np.float64(0.2964580387714544),
 'WMT': np.float64(0.12071022643725152)}

In [123]:
# Sustraer rendimiento del benchmark
alpha_b=rets_b.sub(rets_benchmark_b.squeeze(),axis=0).dropna()
alpha_b

Ticker,CLX,COST,KO,PG,WMT
Date,,,,,
2016-01-05,0.011938,-0.000065,0.001028,0.000681,0.021246
2016-01-06,0.008344,-0.000438,0.003411,-0.000851,0.018828
2016-01-07,0.002124,-0.009843,-0.003474,0.004333,0.036355
2016-01-08,0.012309,-0.009493,0.005367,-0.007667,-0.014902
2016-01-11,0.003251,0.008523,-0.006949,0.000579,0.002066
...,...,...,...,...,...
2026-10-08,-0.006124,-0.015069,0.001636,-0.002347,0.001103
2026-10-09,0.002977,-0.002392,0.001853,0.002913,0.005808
2026-10-12,-0.000707,0.005952,0.012491,0.005757,0.008267


In [124]:
# Calcular target downside risk
below_zero_t_b=(alpha_b[alpha_b<0]).fillna(0)
downside_risk_t_b=np.array(below_zero_t_b.std())
downside_risk_t_b

array([0.00865549, 0.00684588, 0.00473383, 0.00478275, 0.00712828])

In [125]:
# Calcular matriz de target semivariance
semivar_matrix_t_b=(downside_risk_t_b.reshape(n_assets_b,1) @ downside_risk_t_b.reshape(1,n_assets_b))*alpha_b.corr()
semivar_matrix_t_b

Ticker,CLX,COST,KO,PG,WMT
Ticker,,,,,
CLX,7.491748e-05,0.000003,7.313672e-07,0.000011,0.000004
COST,2.607500e-06,0.000047,-2.740260e-06,-0.000001,0.000018
KO,7.313672e-07,-0.000003,2.240916e-05,0.000003,-0.000003
PG,1.118005e-05,-0.000001,2.808965e-06,0.000023,0.000001
WMT,4.002851e-06,0.000018,-3.128373e-06,0.000001,0.000051


In [126]:
# Obtener semivarianza objetivo
func_target_b=lambda w: w.T @ semivar_matrix_t_b @ w
func_target_b(w_target_semivar_b)

np.float64(8.399786258557572e-06)

### 4. COMPARACIÓN DE RESULTADOS

In [127]:
# Comparar los pesos de cada método
pesos_b=pd.DataFrame({
    'Mínima Semivarianza':w_min_semivar_b*100,
    'Máximo Omega':w_max_omega_b*100,
    'Semivarianza Objetivo':w_target_semivar_b*100
},index=prices_b.columns)
pesos_b

,Mínima Semivarianza,Máximo Omega,Semivarianza Objetivo
Ticker,,,
CLX,8.659880,3.351421e-14,5.230520
COST,15.434668,5.290270e-14,15.801241
KO,33.308724,1.790271e-14,37.251412
PG,23.476440,8.881784e-14,29.645804
WMT,19.120288,1.000000e+02,12.071023


In [128]:
# Comparar los valores óptimos
resultados_b=pd.DataFrame({
    'Método':['Mínima Semivarianza','Máximo Omega','Semivarianza Objetivo'],
    'Valor óptimo':[semivar_port_b(w_min_semivar_b),func_omega_b(w_max_omega_b),func_target_b(w_target_semivar_b)]
})
resultados_b['Valor óptimo']=resultados_b['Valor óptimo'].map(lambda x:f'{x:.10f}')
resultados_b

,Método,Valor óptimo
0,Mínima Semivarianza,0.0000354575
1,Máximo Omega,1.0704014710
2,Semivarianza Objetivo,0.0000083998


**Conclusión**  
Al optimizar los cinco activos, en mínima semivarianza obtuvimos 0.0000354575. El mayor peso quedó en KO con 33.31%, seguido de PG con 23.48% y WMT con 19.12%. En máximo Omega el valor fue 1.0704014710 y prácticamente todo el peso quedó en WMT con 100%, entonces este método fue el menos diversificado. Para semivarianza objetivo, utilizando KXI como benchmark, obtuvimos 0.0000083998 y los mayores pesos fueron KO con 37.25% y PG con 29.65%. Podemos ver que los métodos dan distribuciones diferentes porque cada uno busca optimizar algo distinto: reducir pérdidas, maximizar Omega o disminuir el riesgo de quedar por debajo del benchmark.

## PUNTO C

### Diferencias entre TMP y TPMP

La principal diferencia entre la Teoría Moderna y la Teoría Posmoderna de Portafolios es la forma en la que consideran el riesgo.

En la TMP se utiliza la varianza o volatilidad total, entonces se toman en cuenta tanto los movimientos positivos como los negativos. Una ventaja es que podemos analizar qué tanto varían los rendimientos y buscar una mejor relación entre riesgo y rendimiento. Pero una desventaja es que también considera los movimientos positivos como parte del riesgo, aunque para un inversionista tener ganancias mayores a las esperadas no necesariamente es algo malo.

En cambio la TPMP se enfoca más en los rendimientos que quedan por debajo de cero o de algún benchmark. Esto nos parece una ventaja porque como inversionistas normalmente nos preocupa más perder dinero o no alcanzar cierto rendimiento que tener ganancias por encima de lo esperado.

Aunque también tiene desventajas porque los resultados dependen del benchmark o rendimiento objetivo que utilicemos y tampoco significa que por minimizar el riesgo a la baja necesariamente vayamos a obtener el mayor rendimiento.

### ¿Qué método utilizaríamos según el tipo de inversionista?

1. Mínima varianza (TMP): Lo utilizaríamos para un inversionista conservador que busca tener la menor volatilidad posible, aunque eso pueda significar sacrificar rendimiento.

2. Máximo Sharpe (TMP): Para un inversionista que busca una mejor relación entre rendimiento y riesgo y está dispuesto a asumir más volatilidad si el rendimiento esperado lo compensa.

3. Mínima semivarianza (TPMP): Para un inversionista conservador al que le preocupan principalmente las pérdidas. A diferencia de mínima varianza, aquí buscamos reducir el riesgo de obtener rendimientos negativos.

4. Máximo Omega (TPMP): Para un inversionista que busca una mejor relación entre upside y downside risk. Aunque con los resultados de nuestra tarea prácticamente todo el peso quedó en WMT, entonces una desventaja es que este método puede generar portafolios muy concentrados cuando no tenemos restricciones adicionales.

5. Semivarianza objetivo (TPMP): Para un inversionista que busca alcanzar o superar un benchmark. En nuestro caso utilizamos KXI y buscamos minimizar el riesgo de obtener rendimientos por debajo de este ETF.

### Conclusión

Con los resultados podemos ver que los métodos pueden dar ponderaciones bastante diferentes aunque utilicemos activos del mismo sector.

Por ejemplo en máximo Omega prácticamente el 100% quedó en WMT, mientras que en mínima semivarianza los pesos estuvieron más repartidos entre las cinco empresas.

Entonces no podemos decir que un método sea mejor que todos los demás porque cada uno está optimizando algo distinto.

Nosotros consideramos que la TPMP puede ser más útil cuando nos interesa principalmente reducir las pérdidas o no quedar por debajo de un rendimiento objetivo. Pero la TMP también tiene ventajas si queremos analizar la volatilidad total y la relación entre rendimiento y riesgo.

Al final la elección depende del objetivo del inversionista, de cuánto riesgo está dispuesto a asumir y de si le preocupa más la volatilidad en general o específicamente las pérdidas.

## EXTRA

Para el extra seleccionamos FXE, FXC, FXY, FXF y WMT. Utilizamos el método de mínima semivarianza sin benchmark, con rendimientos diarios del periodo del 1 de enero de 2016 al 14 de octubre de 2026. Buscamos la combinación de pesos que reduzca lo más posible las variaciones negativas del portafolio.

In [ ]:
# Descarga de Datos
prices_extra = yf.download(['FXE','FXC','FXY','FXF','WMT'],
                           start='2016-01-01',end='2026-10-15')['Close']
prices_extra.isna().sum()

In [ ]:
# Comprobar datos del periodo
print(prices_extra.index.min(),prices_extra.index.max())
print(prices_extra.isna().sum())

In [ ]:
# Calculo de rendimientos
rets_extra=prices_extra.pct_change().dropna()

In [ ]:
# Calcular matriz de correlación
corr_extra=rets_extra.corr()
corr_extra

In [ ]:
below_zero_extra=(rets_extra[rets_extra<0]).fillna(0)
below_zero_extra

In [ ]:
downside_extra=np.array(below_zero_extra.std())
downside_extra

In [ ]:
# Calcular matriz de semivarianza
n_assets_extra=len(prices_extra.columns)
semivar_matrix_extra=(downside_extra.reshape(n_assets_extra,1) @ downside_extra.reshape(1,n_assets_extra))*corr_extra
semivar_matrix_extra

In [ ]:
semivar_extra=lambda w: w.T @ semivar_matrix_extra @ w

In [ ]:
# Optimización de mínima semivarianza
w_0=np.ones(n_assets_extra)/n_assets_extra
restr=lambda w:sum(w)-1
rangos=[(0,1)]*n_assets_extra

# Escalar la función objetivo para mejorar la precisión numérica
results_extra=minimize(fun=lambda w:semivar_extra(w)*1000000,x0=w_0,
                       method='SLSQP',bounds=rangos,
                       constraints={'fun':restr,'type':'eq'},
                       options={'ftol':1e-12,'maxiter':1000})
w_extra=results_extra.x
print('Optimización correcta:',results_extra.success)
print('Pesos:',dict(zip(prices_extra.columns,w_extra)))
semivar_min_extra=semivar_extra(w_extra)
print('Semivarianza mínima:',semivar_min_extra)

In [ ]:
# Verificar los pesos
print(sum(w_extra))
print((w_extra>0).sum())

### Conclusión

Con FXE, FXC, FXY, FXF y WMT obtuvimos una semivarianza mínima de 0.000004181439. El mayor peso quedó en FXC con 46.36%, seguido de FXF con 21.97%, FXY con 17.10%, FXE con 9.48% y WMT con 5.09%. Los cinco activos tuvieron datos completos en el periodo analizado y la optimización terminó correctamente.

Con esta combinación conseguimos reducir la semivarianza al repartir el portafolio entre activos que no se mueven exactamente de la misma manera. Esto muestra la importancia de tomar en cuenta las correlaciones además del riesgo individual de cada activo.